# Risk and Cloud Explorer

Compare primary-cloud portfolios across risk ratings, market share and adoption maturity

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("data/software_companies_dataset_v2.csv")

# Prepare missing values for reliable widgets and Plotly charts.
categorical_columns = [
    "Company_Name", "Industry", "Headquarters_City", "Country",
    "Ownership_Type", "Customer_Segment", "Primary_Cloud", "Risk_Rating"
]
for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown").astype(str).str.strip()

numeric_columns = [
    "Employees", "Annual_Revenue", "Profit_Margin", "Market_Share",
    "R&D_Spending", "Average_Salary", "Training_Hours_Per_Employee",
    "Employee_Satisfaction", "Adoption_Rate_AI", "Adoption_Rate_Cloud",
    "Adoption_Rate_Blockchain"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        df[column] = df[column].fillna(df[column].median())

df["Annual_Revenue"] = df["Annual_Revenue"].clip(lower=1)
df["Employees"] = df["Employees"].clip(lower=1)

risk = widgets.SelectMultiple(options=sorted(df["Risk_Rating"].dropna().astype(str).unique().tolist()), description="Risk")
cloud = widgets.SelectMultiple(options=sorted(df["Primary_Cloud"].dropna().astype(str).unique().tolist()), description="Cloud")
out = widgets.Output()

def render(*_):
    view = df.copy()
    if risk.value: view = view[view["Risk_Rating"].isin(risk.value)]
    if cloud.value: view = view[view["Primary_Cloud"].isin(cloud.value)]
    with out:
        clear_output(wait=True)
        agg = view.groupby(["Primary_Cloud","Risk_Rating"], as_index=False).agg(
            Companies=("Company_ID","count"), Revenue=("Annual_Revenue","sum"))
        px.sunburst(agg, path=["Primary_Cloud","Risk_Rating"], values="Revenue",
                    color="Companies", title="Cloud portfolio and risk exposure").show()
        px.scatter(view, x="Adoption_Rate_Cloud", y="Market_Share",
                   size="Annual_Revenue", color="Risk_Rating",
                   hover_name="Company_Name", title="Cloud adoption versus market share").show()

risk.observe(render, names="value")
cloud.observe(render, names="value")
display(widgets.HBox([risk, cloud]), out)
render()